# Adult Census Dataset - Data Cleaning Pipeline

This notebook cleans the Adult Census dataset for income classification task.
Dataset source: UCI Machine Learning Repository

In [105]:
import pandas as pd
import warnings
from sklearn.preprocessing import LabelEncoder
warnings.filterwarnings('ignore')

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

## 1. Data Loading and Initial Exploration

In [106]:
# Define column names based on adult.names file
column_names = [
    'age', 'workclass', 'fnlwgt', 'education', 'education_num',
    'marital_status', 'occupation', 'relationship', 'race', 'sex',
    'capital_gain', 'capital_loss', 'hours_per_week', 'native_country', 'income'
]

# Load training data
train_data = pd.read_csv('adult_database/adult.data', 
                        names=column_names, 
                        skipinitialspace=True,
                        na_values='?')

# Load test data
test_data = pd.read_csv('adult_database/adult.test', 
                       names=column_names, 
                       skipinitialspace=True,
                       na_values='?',
                       skiprows=1)  # Skip the first line in test file

print(f"Training data shape: {train_data.shape}")
print(f"Test data shape: {test_data.shape}")
print(f"Total records: {train_data.shape[0] + test_data.shape[0]}")

Training data shape: (32561, 15)
Test data shape: (16281, 15)
Total records: 48842


In [107]:
# Add source identifier before combining
train_data['source'] = 'train'
test_data['source'] = 'test'

# Combine datasets for unified cleaning
combined_data = pd.concat([train_data, test_data], ignore_index=True)

print(f"Combined dataset shape: {combined_data.shape}")
print("\nFirst few rows:")
combined_data.head()

Combined dataset shape: (48842, 16)

First few rows:


,age,workclass,fnlwgt,education,education_num,marital_status,occupation,relationship,race,sex,capital_gain,capital_loss,hours_per_week,native_country,income,source
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K,train
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K,train
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K,train
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K,train
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K,train


## 2. Data Quality Assessment

In [108]:
# Check data types and basic info
print("Dataset Info:")
print(combined_data.info())

print("\nData Types:")
print(combined_data.dtypes)

print("\nBasic Statistics:")
print(combined_data.describe())

Dataset Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 48842 entries, 0 to 48841
Data columns (total 16 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   age             48842 non-null  int64 
 1   workclass       46043 non-null  object
 2   fnlwgt          48842 non-null  int64 
 3   education       48842 non-null  object
 4   education_num   48842 non-null  int64 
 5   marital_status  48842 non-null  object
 6   occupation      46033 non-null  object
 7   relationship    48842 non-null  object
 8   race            48842 non-null  object
 9   sex             48842 non-null  object
 10  capital_gain    48842 non-null  int64 
 11  capital_loss    48842 non-null  int64 
 12  hours_per_week  48842 non-null  int64 
 13  native_country  47985 non-null  object
 14  income          48842 non-null  object
 15  source          48842 non-null  object
dtypes: int64(6), object(10)
memory usage: 6.0+ MB
None

Data Types:
age             

In [109]:
# Check for missing values
print("Missing Values Count:")
missing_counts = combined_data.isnull().sum()
missing_percentage = (missing_counts / len(combined_data)) * 100
missing_df = pd.DataFrame({
    'Missing Count': missing_counts,
    'Percentage': missing_percentage
})
print(missing_df[missing_df['Missing Count'] > 0])

Missing Values Count:
                Missing Count  Percentage
workclass                2799    5.730724
occupation               2809    5.751198
native_country            857    1.754637


In [110]:
# Check unique values for categorical columns
categorical_columns = ['workclass', 'education', 'marital_status', 'occupation', 
                      'relationship', 'race', 'sex', 'native_country', 'income']

print("Unique values in categorical columns:")
for col in categorical_columns:
    print(f"\n{col}: {combined_data[col].nunique()} unique values")
    print(combined_data[col].value_counts().head(10))

Unique values in categorical columns:

workclass: 8 unique values
workclass
Private             33906
Self-emp-not-inc     3862
Local-gov            3136
State-gov            1981
Self-emp-inc         1695
Federal-gov          1432
Without-pay            21
Never-worked           10
Name: count, dtype: int64

education: 16 unique values
education
HS-grad         15784
Some-college    10878
Bachelors        8025
Masters          2657
Assoc-voc        2061
11th             1812
Assoc-acdm       1601
10th             1389
7th-8th           955
Prof-school       834
Name: count, dtype: int64

marital_status: 7 unique values
marital_status
Married-civ-spouse       22379
Never-married            16117
Divorced                  6633
Separated                 1530
Widowed                   1518
Married-spouse-absent      628
Married-AF-spouse           37
Name: count, dtype: int64

occupation: 14 unique values
occupation
Prof-specialty       6172
Craft-repair         6112
Exec-managerial      

## 3. Data Cleaning and Preprocessing

In [111]:
# Create a copy for cleaning
df_clean = combined_data.copy()

# 1. Clean target variable (income)
# Remove trailing periods from income labels
df_clean['income'] = df_clean['income'].str.replace('.', '', regex=False)
print("Income distribution after cleaning:")
print(df_clean['income'].value_counts())

# Convert to binary target (1 for >50K, 0 for <=50K)
df_clean['income_binary'] = (df_clean['income'] == '>50K').astype(int)
print("\nBinary income distribution:")
print(df_clean['income_binary'].value_counts())

Income distribution after cleaning:
income
<=50K    37155
>50K     11687
Name: count, dtype: int64

Binary income distribution:
income_binary
0    37155
1    11687
Name: count, dtype: int64


In [112]:
# 2. Handle missing values
print("Handling missing values...")

# For workclass: replace missing with mode
workclass_mode = df_clean['workclass'].mode()[0]
df_clean['workclass'].fillna(workclass_mode, inplace=True)

# For occupation: replace missing with mode
occupation_mode = df_clean['occupation'].mode()[0]
df_clean['occupation'].fillna(occupation_mode, inplace=True)

# For native_country: replace missing with mode (United-States is most common)
native_country_mode = df_clean['native_country'].mode()[0]
df_clean['native_country'].fillna(native_country_mode, inplace=True)

print("Missing values after handling:")
print(df_clean.isnull().sum().sum())

Handling missing values...
Missing values after handling:
0


In [113]:
# 3. Feature Engineering
print("Creating engineered features...")

# Age groups
df_clean['age_group'] = pd.cut(df_clean['age'], 
                              bins=[0, 25, 35, 45, 55, 65, 100],
                              labels=['18-25', '26-35', '36-45', '46-55', '56-65', '65+'])

# Capital features
df_clean['capital_net'] = df_clean['capital_gain'] - df_clean['capital_loss']
df_clean['has_capital_gain'] = (df_clean['capital_gain'] > 0).astype(int)
df_clean['has_capital_loss'] = (df_clean['capital_loss'] > 0).astype(int)

# Work hours categories
df_clean['hours_category'] = pd.cut(df_clean['hours_per_week'],
                                   bins=[0, 20, 40, 60, 100],
                                   labels=['part_time', 'full_time', 'overtime', 'extreme'])

# Education level grouping
education_mapping = {
    'Preschool': 'Elementary',
    '1st-4th': 'Elementary',
    '5th-6th': 'Elementary',
    '7th-8th': 'Middle',
    '9th': 'High School',
    '10th': 'High School',
    '11th': 'High School',
    '12th': 'High School',
    'HS-grad': 'High School',
    'Some-college': 'College',
    'Assoc-acdm': 'College',
    'Assoc-voc': 'College',
    'Bachelors': 'Bachelor',
    'Masters': 'Graduate',
    'Prof-school': 'Graduate',
    'Doctorate': 'Graduate'
}
df_clean['education_level'] = df_clean['education'].map(education_mapping)

print("Feature engineering completed.")

Creating engineered features...
Feature engineering completed.


## 4. Final Dataset Preparation

In [114]:
# Select relevant features for ML model
feature_columns = [
    'age', 'workclass', 'fnlwgt', 'education', 'education_num',
    'marital_status', 'occupation', 'relationship', 'race', 'sex',
    'capital_gain', 'capital_loss', 'hours_per_week', 'native_country',
    'age_group', 'capital_net', 'has_capital_gain', 'has_capital_loss',
    'hours_category', 'education_level', 'income_binary', 'source'
]

# Create final cleaned dataset
df_final = df_clean[feature_columns].copy()

print(f"Final dataset shape: {df_final.shape}")
print("\nFinal dataset info:")
print(df_final.info())

# Check for any remaining missing values
print("\nRemaining missing values:")
print(df_final.isnull().sum())

Final dataset shape: (48842, 22)

Final dataset info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 48842 entries, 0 to 48841
Data columns (total 22 columns):
 #   Column            Non-Null Count  Dtype   
---  ------            --------------  -----   
 0   age               48842 non-null  int64   
 1   workclass         48842 non-null  object  
 2   fnlwgt            48842 non-null  int64   
 3   education         48842 non-null  object  
 4   education_num     48842 non-null  int64   
 5   marital_status    48842 non-null  object  
 6   occupation        48842 non-null  object  
 7   relationship      48842 non-null  object  
 8   race              48842 non-null  object  
 9   sex               48842 non-null  object  
 10  capital_gain      48842 non-null  int64   
 11  capital_loss      48842 non-null  int64   
 12  hours_per_week    48842 non-null  int64   
 13  native_country    48842 non-null  object  
 14  age_group         48842 non-null  category
 15  capital_net     

In [115]:
# Create separate train and test datasets
train_final = df_final[df_final['source'] == 'train'].drop('source', axis=1).copy()
test_final = df_final[df_final['source'] == 'test'].drop('source', axis=1).copy()

print(f"Training set shape: {train_final.shape}")
print(f"Test set shape: {test_final.shape}")

# Verify income distribution in both sets
print("\nTraining set income distribution:")
print(train_final['income_binary'].value_counts(normalize=True))

print("\nTest set income distribution:")
print(test_final['income_binary'].value_counts(normalize=True))

Training set shape: (32561, 21)
Test set shape: (16281, 21)

Training set income distribution:
income_binary
0    0.75919
1    0.24081
Name: proportion, dtype: float64

Test set income distribution:
income_binary
0    0.763774
1    0.236226
Name: proportion, dtype: float64


In [116]:
# We should turn the categorical data into binary here:
df_final.head(5)

# Convert to binary sex (1 for Male, 0 for Female)
df_final['sex'] = (df_final['sex'] == 'Male').astype(int)
print("\nBinary sex distribution:")
print(df_final['sex'].value_counts())
# Convert to binary Relationship (1 for family and 0 for unmarried/not in family)

df_final['relationship_binary'] = df_final['relationship'].apply(
    lambda x: 1 if x in ['Husband', 'Wife', 'Own-child', 'Other-relative'] else 0
)

df_final.head(5)
# convert to a labelel encoding( 0 for married, 1 for single and 2 for previously married)

df_final['marital_status'] = df_final['marital_status'].apply(
    lambda x: 0 if x in ['Married-civ-spouse', 'Married-AF-spouse']
    else 1 if x == 'Never-married'
    else 2
)
df_final.head(5)

# Hours per week, note: drop hours category column since we already have a numerical value for hours
# Education level note: drop either Education_level or Education, probably best o take Education
# Workclass


Binary sex distribution:
sex
1    32650
0    16192
Name: count, dtype: int64


,age,workclass,fnlwgt,education,education_num,marital_status,occupation,relationship,race,sex,capital_gain,capital_loss,hours_per_week,native_country,age_group,capital_net,has_capital_gain,has_capital_loss,hours_category,education_level,income_binary,source,relationship_binary
0,39,State-gov,77516,Bachelors,13,1,Adm-clerical,Not-in-family,White,1,2174,0,40,United-States,36-45,2174,1,0,full_time,Bachelor,0,train,0
1,50,Self-emp-not-inc,83311,Bachelors,13,0,Exec-managerial,Husband,White,1,0,0,13,United-States,46-55,0,0,0,part_time,Bachelor,0,train,1
2,38,Private,215646,HS-grad,9,2,Handlers-cleaners,Not-in-family,White,1,0,0,40,United-States,36-45,0,0,0,full_time,High School,0,train,0
3,53,Private,234721,11th,7,0,Handlers-cleaners,Husband,Black,1,0,0,40,United-States,46-55,0,0,0,full_time,High School,0,train,1
4,28,Private,338409,Bachelors,13,0,Prof-specialty,Wife,Black,0,0,0,40,Cuba,26-35,0,0,0,full_time,Bachelor,0,train,1


In [117]:
df_final = df_final.drop (columns = ['hours_category' , 'education_num', 'education', 'workclass', 'relationship'])
df_final.head(5)

,age,fnlwgt,marital_status,occupation,race,sex,capital_gain,capital_loss,hours_per_week,native_country,age_group,capital_net,has_capital_gain,has_capital_loss,education_level,income_binary,source,relationship_binary
0,39,77516,1,Adm-clerical,White,1,2174,0,40,United-States,36-45,2174,1,0,Bachelor,0,train,0
1,50,83311,0,Exec-managerial,White,1,0,0,13,United-States,46-55,0,0,0,Bachelor,0,train,1
2,38,215646,2,Handlers-cleaners,White,1,0,0,40,United-States,36-45,0,0,0,High School,0,train,0
3,53,234721,0,Handlers-cleaners,Black,1,0,0,40,United-States,46-55,0,0,0,High School,0,train,1
4,28,338409,0,Prof-specialty,Black,0,0,0,40,Cuba,26-35,0,0,0,Bachelor,0,train,1


In [118]:
# Use get_dummies for race column to create binary columns
race_dummies = pd.get_dummies(df_final['race'], prefix='race')
df_final = pd.concat([df_final, race_dummies], axis=1)

# Drop the original race column
df_final = df_final.drop('race', axis=1)

print("Race columns after one-hot encoding:")
print([col for col in df_final.columns if col.startswith('race_')])
print(f"\nDataframe shape after adding race dummies: {df_final.shape}")
df_final.head()

Race columns after one-hot encoding:
['race_Amer-Indian-Eskimo', 'race_Asian-Pac-Islander', 'race_Black', 'race_Other', 'race_White']

Dataframe shape after adding race dummies: (48842, 22)


,age,fnlwgt,marital_status,occupation,sex,capital_gain,capital_loss,hours_per_week,native_country,age_group,capital_net,has_capital_gain,has_capital_loss,education_level,income_binary,source,relationship_binary,race_Amer-Indian-Eskimo,race_Asian-Pac-Islander,race_Black,race_Other,race_White
0,39,77516,1,Adm-clerical,1,2174,0,40,United-States,36-45,2174,1,0,Bachelor,0,train,0,False,False,False,False,True
1,50,83311,0,Exec-managerial,1,0,0,13,United-States,46-55,0,0,0,Bachelor,0,train,1,False,False,False,False,True
2,38,215646,2,Handlers-cleaners,1,0,0,40,United-States,36-45,0,0,0,High School,0,train,0,False,False,False,False,True
3,53,234721,0,Handlers-cleaners,1,0,0,40,United-States,46-55,0,0,0,High School,0,train,1,False,False,True,False,False
4,28,338409,0,Prof-specialty,0,0,0,40,Cuba,26-35,0,0,0,Bachelor,0,train,1,False,False,True,False,False


In [119]:
# Use label encoder for education_level to assign specific numbers

# Create label encoder
le_education = LabelEncoder()

# Apply label encoding to education_level
df_final['education_level_encoded'] = le_education.fit_transform(df_final['education_level'])

# Display the mapping
education_mapping_encoded = dict(zip(le_education.classes_, le_education.transform(le_education.classes_)))
print("Education level encoding mapping:")
for level, code in education_mapping_encoded.items():
    print(f"{level}: {code}")

# Show the updated dataframe
print(f"\nDataframe shape after encoding: {df_final.shape}")
print("\nEducation level distribution:")
print(df_final['education_level_encoded'].value_counts().sort_index())
df_final = df_final.drop('education_level', axis=1)

Education level encoding mapping:
Bachelor: 0
College: 1
Elementary: 2
Graduate: 3
High School: 4
Middle: 5

Dataframe shape after encoding: (48842, 23)

Education level distribution:
education_level_encoded
0     8025
1    14540
2      839
3     4085
4    20398
5      955
Name: count, dtype: int64


In [120]:
df_final.head(5)

,age,fnlwgt,marital_status,occupation,sex,capital_gain,capital_loss,hours_per_week,native_country,age_group,capital_net,has_capital_gain,has_capital_loss,income_binary,source,relationship_binary,race_Amer-Indian-Eskimo,race_Asian-Pac-Islander,race_Black,race_Other,race_White,education_level_encoded
0,39,77516,1,Adm-clerical,1,2174,0,40,United-States,36-45,2174,1,0,0,train,0,False,False,False,False,True,0
1,50,83311,0,Exec-managerial,1,0,0,13,United-States,46-55,0,0,0,0,train,1,False,False,False,False,True,0
2,38,215646,2,Handlers-cleaners,1,0,0,40,United-States,36-45,0,0,0,0,train,0,False,False,False,False,True,4
3,53,234721,0,Handlers-cleaners,1,0,0,40,United-States,46-55,0,0,0,0,train,1,False,False,True,False,False,4
4,28,338409,0,Prof-specialty,0,0,0,40,Cuba,26-35,0,0,0,0,train,1,False,False,True,False,False,0


## 5. Save Cleaned Data

In [121]:
# Save cleaned datasets
train_final.to_csv('cleaned_data/adult_train_cleaned.csv', index=False)
test_final.to_csv('cleaned_data/adult_test_cleaned.csv', index=False)
df_final.to_csv('cleaned_data/adult_combined_cleaned.csv', index=False)

print("Cleaned datasets saved successfully!")
print("\nFiles created:")
print("- cleaned_data/adult_train_cleaned.csv")
print("- cleaned_data/adult_test_cleaned.csv")
print("- cleaned_data/adult_combined_cleaned.csv")

Cleaned datasets saved successfully!

Files created:
- cleaned_data/adult_train_cleaned.csv
- cleaned_data/adult_test_cleaned.csv
- cleaned_data/adult_combined_cleaned.csv


## 6. Data Summary Repor

In [122]:
# Generate summary report
summary_report = {
    'Original Dataset': {
        'Total Records': len(combined_data),
        'Training Records': len(train_data),
        'Test Records': len(test_data),
        'Features': len(combined_data.columns) - 1,  # Excluding source column
        'Missing Values': combined_data.isnull().sum().sum()
    },
    'Cleaned Dataset': {
        'Total Records': len(df_final),
        'Training Records': len(train_final),
        'Test Records': len(test_final),
        'Features': len(df_final.columns) - 1,  # Excluding source column
        'Missing Values': df_final.isnull().sum().sum()
    },
    'Feature Engineering': {
        'New Features Created': 6,
        'Features Added': ['age_group', 'capital_net', 'has_capital_gain', 
                          'has_capital_loss', 'hours_category', 'education_level']
    },
    'Data Quality': {
        'Duplicates Removed': 0,
        'Missing Values Handled': 'Yes',
        'Target Variable': 'income_binary (0: <=50K, 1: >50K)',
        'Class Distribution': f"{df_final['income_binary'].value_counts(normalize=True)[0]:.2%} / {df_final['income_binary'].value_counts(normalize=True)[1]:.2%}"
    }
}

print("=" * 50)
print("DATA CLEANING SUMMARY REPORT")
print("=" * 50)

for section, details in summary_report.items():
    print(f"\n{section}:")
    print("-" * 30)
    for key, value in details.items():
        print(f"{key}: {value}")

print("\n" + "=" * 50)
print("READY FOR MACHINE LEARNING MODELS")
print("=" * 50)

DATA CLEANING SUMMARY REPORT

Original Dataset:
------------------------------
Total Records: 48842
Training Records: 32561
Test Records: 16281
Features: 15
Missing Values: 6465

Cleaned Dataset:
------------------------------
Total Records: 48842
Training Records: 32561
Test Records: 16281
Features: 21
Missing Values: 0

Feature Engineering:
------------------------------
New Features Created: 6
Features Added: ['age_group', 'capital_net', 'has_capital_gain', 'has_capital_loss', 'hours_category', 'education_level']

Data Quality:
------------------------------
Duplicates Removed: 0
Missing Values Handled: Yes
Target Variable: income_binary (0: <=50K, 1: >50K)
Class Distribution: 76.07% / 23.93%

READY FOR MACHINE LEARNING MODELS
